## Setup

In [1]:
# =============================================================================
# Wild Boar Functional Connectivity Pipeline
# Methods: Graph Theory + Circuitscape (Pinch-points) + Least-Cost Corridors
# Based on: Urbina et al. (in-review) approach as applied in Rothirsch Mittelland
# =============================================================================
# REQUIREMENTS:
#   R packages: terra, sf, igraph, gdistance, dplyr, ggplot2
#   External:   Circuitscape via Julia (julia + Circuitscape.jl installed)
#   Input 1:    Resistance raster  -> ../data/processed/Resistance_Keeley_Aargau_25m.tif
#   Input 2:    Core habitat patches (Kerngebiete) -> set path in CONFIG below
# =============================================================================

# --------------------------------------------- 0. PACKAGES & CONFIG ----------

install.packages(c("terra", "sf", "igraph", "gdistance", "spdep", "dplyr", "ggplot2"))

library(terra)      # raster/vector operations
library(sf)         # vector operations
library(igraph)     # landscape graph
library(gdistance)  # least-cost paths & cumulative costs
library(dplyr)
library(ggplot2)

# ── USER CONFIG ───────────────────────────────────────────────────────────────
RESISTANCE_PATH  <- "../data/processed/Resistance_Keeley_Aargau_25m.tif"
CORE_PATCHES_PATH <- "../data/processed/Core_Habitats_50ha_Nodes.shp"
OUT_DIR          <- "../data/processed/connectivity/"
MAX_DIST_KM      <- 30        # max Euclidean distance for graph edges (km)
TILE_SIZE_CELLS  <- 400       # Circuitscape tile size (pixels); adjust to RAM
JULIA_PATH       <- "C:/Users/Lukas/AppData/Local/Programs/Julia-1.12.6/bin/julia.exe"   # path to julia executable, e.g. "/usr/local/bin/julia"
# ─────────────────────────────────────────────────────────────────────────────

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(OUT_DIR, "circuitscape_tiles"), recursive = TRUE, showWarnings = FALSE)




Installing packages into ‘C:/Users/Lukas/AppData/Local/R/win-library/4.5’
(as ‘lib’ is unspecified)


trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/terra_1.9-11.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/sf_1.1-0.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/igraph_2.2.3.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/gdistance_1.6.5.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/spdep_1.4-2.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/dplyr_1.2.1.zip'
trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/ggplot2_4.0.2.zip'


package ‘terra’ successfully unpacked and MD5 sums checked
package ‘sf’ successfully unpacked and MD5 sums checked
package ‘igraph’ successfully unpacked and MD5 sums checked
package ‘gdistance’ successfully unpacked and MD5 sums checked
package ‘spdep’ successfully unpacked and MD5 sums checked
package ‘dplyr’ successfully unpacked and MD5 sums checked
package ‘ggplot2’ successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Lukas\AppData\Local\Temp\RtmpyQpfz2\downloaded_packages
terra 1.9.11


Warning message:
package ‘terra’ was built under R version 4.5.3 


Linking to GEOS 3.14.1, GDAL 3.12.1, PROJ 9.7.1; sf_use_s2() is TRUE


...

Warning message:
package ‘igraph’ was built under R version 4.5.3 


Loading required package: raster
Loading required package: sp
Loading required package: Matrix

Attaching package: ‘gdistance’

The following object is masked from ‘package:igraph’:

    normalize



Warning messages:
1: package ‘gdistance’ was built under R version 4.5.3 
2: package ‘raster’ was built under R version 4.5.2 
3: package ‘sp’ was built under R version 4.5.2 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:raster’:

    intersect, select, union

The following objects are masked from ‘package:igraph’:

    as_data_frame, groups, union

The following objects are masked from ‘package:terra’:

    intersect, union

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Warning message:
package ‘dplyr’ was built under R version 4.5.3 
Warning message:
package ‘ggplot2’ was built under R version 4.5.3 


In [2]:
# --------------------------------------------- 1. LOAD INPUTS ----------------

cat("── 1. Loading inputs ──\n")
resistance <- rast(RESISTANCE_PATH)
cores_sf   <- st_read(CORE_PATCHES_PATH, quiet = TRUE)

# Reproject cores to match resistance raster CRS if needed
cores_sf <- st_transform(cores_sf, crs = crs(resistance))

# Add unique ID column if not present
if (!"patch_id" %in% names(cores_sf)) {
  cores_sf$patch_id <- seq_len(nrow(cores_sf))
}

# Compute centroids for graph nodes
centroids <- st_centroid(cores_sf)
coords    <- st_coordinates(centroids)   # matrix [x, y]

cat(sprintf("  Resistance raster: %d rows x %d cols, res = %.1f m\n",
            nrow(resistance), ncol(resistance), res(resistance)[1]))
cat(sprintf("  Core patches loaded: %d patches\n", nrow(cores_sf)))




── 1. Loading inputs ──


Warning message:
st_centroid assumes attributes are constant over geometries 


  Resistance raster: 2144 rows x 2245 cols, res = 25.0 m
  Core patches loaded: 196 patches


## Landscape diagram

In [3]:
# --------------------------------------------- 2. LANDSCAPE GRAPH ------------
# Planar graph between core patch centroids with max Euclidean distance of 30 km
# Equivalent to Graphab planar graph approach (Foltête et al. 2021)

cat("\n── 2. Building landscape graph ──\n")

# Euclidean distance matrix between centroids (in metres)
dist_mat <- as.matrix(dist(coords))

# Build planar graph: connect each node to its nearest neighbours within MAX_DIST_KM
# using a Delaunay triangulation to avoid edge crossings (planar approximation)
max_dist_m <- MAX_DIST_KM * 1000

# Delaunay triangulation via sf for planarity
library(spdep)   # for tri2nb
nb <- tri2nb(coords)   # Delaunay triangulation neighbour list

# Convert to edge list, filter by max distance
edges <- do.call(rbind, lapply(seq_along(nb), function(i) {
  j_vec <- nb[[i]]
  j_vec <- j_vec[j_vec > i]  # upper triangle only
  if (length(j_vec) == 0) return(NULL)
  data.frame(from = i, to = j_vec,
             dist_m = dist_mat[i, j_vec])
}))
edges <- edges[edges$dist_m <= max_dist_m, ]

cat(sprintf("  Graph edges after distance filter: %d\n", nrow(edges)))

# Create igraph object
g <- graph_from_data_frame(edges, directed = FALSE,
                           vertices = data.frame(id = seq_len(nrow(cores_sf)),
                                                 x  = coords[, 1],
                                                 y  = coords[, 2]))

# Save graph edge list (node pairs used in steps 3 & 4)
write.csv(edges, file.path(OUT_DIR, "graph_edges.csv"), row.names = FALSE)

# Save graph plot
png(file.path(OUT_DIR, "landscape_graph.png"), width = 1800, height = 1600, res = 200)
plot(st_geometry(cores_sf), col = "#2d6a4f55", border = "#2d6a4f",
     main = "Landscape Graph – Wild Boar Core Patches")
for (i in seq_len(nrow(edges))) {
  lines(rbind(coords[edges$from[i], ], coords[edges$to[i], ]),
        col = "#e76f51", lwd = 1.2)
}
points(coords, pch = 21, bg = "#2d6a4f", col = "white", cex = 1.5)
dev.off()
cat("  Graph saved to landscape_graph.png\n")





── 2. Building landscape graph ──
Loading required package: spData
To access larger datasets in this package, install the spDataLarge package with:
`install.packages('spDataLarge', repos='https://nowosad.github.io/drat/', type='source')`

Attaching package: ‘spData’

The following object is masked _by_ ‘.GlobalEnv’:

    coords



Warning messages:
1: package ‘spdep’ was built under R version 4.5.3 
2: package ‘spData’ was built under R version 4.5.3 


  Graph edges after distance filter: 572
  Graph saved to landscape_graph.png


## Circuitscape Pinch-Points

In [4]:
cat("Installiere Circuitscape in Julia (das kann 1-2 Minuten dauern)...\n")

# Wir senden den Pkg.add() Befehl an Julia
cmd_install <- sprintf('%s -e "using Pkg; Pkg.add(\\"Circuitscape\\")"', JULIA_PATH)

# Ausführen - diesmal lassen wir den Output anzeigen, damit wir den Fortschritt sehen
system(cmd_install, ignore.stdout = FALSE, ignore.stderr = FALSE)

cat("\nInstallation abgeschlossen! Du kannst jetzt deinen Tile-Code neu starten.\n")

Installiere Circuitscape in Julia (das kann 1-2 Minuten dauern)...
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\Lukas\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\Lukas\.julia\environments\v1.12\Manifest.toml`

Installation abgeschlossen! Du kannst jetzt deinen Tile-Code neu starten.


In [5]:
# --------------------------------------------- 3. CIRCUITSCAPE PINCH-POINTS -
# Tile-based approach (Pelletier et al. 2014):
#   - Divide study area into overlapping tiles (100% buffer)
#   - Run Circuitscape N↔S and E↔W for each tile
#   - Mosaic results → current density map

cat("\n── 3. Circuitscape pinch-point analysis (tile approach) ──\n")

# Helper: write a Circuitscape .ini config file
write_cs_ini <- function(ini_path, resistance_asc, output_dir,
                         node_file, scenario = "all-to-one") {
  ini_text <- sprintf(
'[circuitscape options]
data_type = raster
scenario = pairwise
write_cur_maps = true
write_cum_cur_map_only = false
log_level = ERROR
print_timings = false

[Habitat raster or graph]
habitat_file = %s
habitat_map_is_resistances = true

[Options for pairwise and one-to-all and all-to-one modes]
point_file = %s

[Output options]
output_file = %s
', resistance_asc,
   node_file,
   file.path(output_dir, "cs_output.out"))
  writeLines(ini_text, ini_path)
}

# Helper: write ASCII raster
write_asc <- function(r, path) {
  writeRaster(r, path, filetype = "AAIGrid", overwrite = TRUE)
}

# Helper: write a 1-pixel wide node strip as point file
write_node_file <- function(r, direction = "NS", path) {
  ext_r <- ext(r)
  res_r <- res(r)[1]
  if (direction == "NS") {
    # Top row (North) and bottom row (South) centroids
    x_seq <- seq(ext_r$xmin + res_r/2, ext_r$xmax - res_r/2, by = res_r)
    north_pts <- data.frame(x = x_seq,
                            y = rep(ext_r$ymax - res_r/2, length(x_seq)),
                            id = 1)
    south_pts <- data.frame(x = x_seq,
                            y = rep(ext_r$ymin + res_r/2, length(x_seq)),
                            id = 2)
    pts <- rbind(north_pts, south_pts)
  } else {  # EW
    y_seq <- seq(ext_r$ymin + res_r/2, ext_r$ymax - res_r/2, by = res_r)
    east_pts  <- data.frame(x = rep(ext_r$xmax - res_r/2, length(y_seq)),
                            y = y_seq, id = 1)
    west_pts  <- data.frame(x = rep(ext_r$xmin + res_r/2, length(y_seq)),
                            y = y_seq, id = 2)
    pts <- rbind(east_pts, west_pts)
  }
  write.table(pts[, c("id", "x", "y")], path,
              col.names = FALSE, row.names = FALSE, sep = " ")
}

# ── Tile grid ─────────────────────────────────────────────────────────────────
ext_full <- ext(resistance)
res_m    <- res(resistance)[1]
tile_m   <- TILE_SIZE_CELLS * res_m
buf_m    <- tile_m  # 100% buffer

x_starts <- seq(ext_full$xmin, ext_full$xmax, by = tile_m)
y_starts <- seq(ext_full$ymin, ext_full$ymax, by = tile_m)

tile_results_NS <- list()
tile_results_EW <- list()
tile_idx <- 0

for (xi in seq_along(x_starts)) {
  for (yi in seq_along(y_starts)) {
    tile_idx <- tile_idx + 1
    tile_dir  <- file.path(OUT_DIR, "circuitscape_tiles", sprintf("tile_%03d", tile_idx))
    dir.create(tile_dir, showWarnings = FALSE)

    # Tile extent (core tile, no buffer)
    tile_ext <- ext(
      x_starts[xi],
      min(x_starts[xi] + tile_m, ext_full$xmax),
      y_starts[yi],
      min(y_starts[yi] + tile_m, ext_full$ymax)
    )

    # Buffered computation extent (100% buffer)
    comp_ext <- ext(
      max(tile_ext$xmin - buf_m, ext_full$xmin),
      min(tile_ext$xmax + buf_m, ext_full$xmax),
      max(tile_ext$ymin - buf_m, ext_full$ymin),
      min(tile_ext$ymax + buf_m, ext_full$ymax)
    )

    # Crop resistance to buffered extent
    r_tile <- crop(resistance, comp_ext)

    # Skip tiles with all NA
    if (all(is.na(values(r_tile)))) next

    asc_path <- file.path(tile_dir, "resistance.asc")
    write_asc(r_tile, asc_path)

    # ── North-South run ────────────────────────────────────────────────────
    node_NS  <- file.path(tile_dir, "nodes_NS.txt")
    ini_NS   <- file.path(tile_dir, "cs_NS.ini")
    write_node_file(r_tile, "NS", node_NS)
    write_cs_ini(ini_NS, asc_path, tile_dir, node_NS)

    # Run Circuitscape via Julia
    cmd_NS <- sprintf('%s -e "using Circuitscape; compute(\\"%s\\")"', JULIA_PATH, ini_NS)
    system(cmd_NS, ignore.stdout = FALSE, ignore.stderr = FALSE)

    out_NS <- file.path(tile_dir, "cs_output_cum_curmap.asc")
    if (file.exists(out_NS)) {
      r_NS <- rast(out_NS)
      r_NS <- crop(r_NS, tile_ext)   # trim buffer
      tile_results_NS[[tile_idx]] <- r_NS
    }

    # ── East-West run ──────────────────────────────────────────────────────
    node_EW  <- file.path(tile_dir, "nodes_EW.txt")
    ini_EW   <- file.path(tile_dir, "cs_EW.ini")
    write_node_file(r_tile, "EW", node_EW)
    write_cs_ini(ini_EW, asc_path, tile_dir, node_EW)

    cmd_EW <- sprintf('%s -e "using Circuitscape; compute(\\"%s\\")"', JULIA_PATH, ini_EW)
    system(cmd_EW, ignore.stdout = FALSE, ignore.stderr = FALSE)

    out_EW <- file.path(tile_dir, "cs_output_cum_curmap.asc")
    if (file.exists(out_EW)) {
      r_EW <- rast(out_EW)
      r_EW <- crop(r_EW, tile_ext)
      tile_results_EW[[tile_idx]] <- r_EW
    }

    cat(sprintf("  Tile %d/%d processed\n", tile_idx,
                length(x_starts) * length(y_starts)))
  }
}

# ── Sum NS + EW per tile, mosaic all tiles ─────────────────────────────────
cat("  Mosaicking tiles...\n")

combined_tiles <- lapply(seq_along(tile_results_NS), function(i) {
  if (!is.null(tile_results_NS[[i]]) && !is.null(tile_results_EW[[i]])) {
    # Bilineares Resampling, falls die Extents minimal (Rundungsfehler) abweichen
    resample(tile_results_EW[[i]], tile_results_NS[[i]], method = "bilinear") + 
      tile_results_NS[[i]]
  } else if (!is.null(tile_results_NS[[i]])) {
    tile_results_NS[[i]]
  } else if (!is.null(tile_results_EW[[i]])) {
    tile_results_EW[[i]]
  } else {
    NULL
  }
})

# Alle NULL-Werte entfernen
combined_tiles <- combined_tiles[!sapply(combined_tiles, is.null)]

# --- SICHERHEITS-CHECK ---
if (length(combined_tiles) == 0) {
  stop("FEHLER: Es wurden keine Kacheln berechnet! Julia ist wahrscheinlich im Hintergrund abgestürzt. Überprüfe die tile-Ordner, ob dort .asc Dateien liegen.")
}

cat(sprintf("  Erstelle Mosaik aus %d erfolgreichen Kacheln...\n", length(combined_tiles)))

# --- DER TERRA-FIX ---
# In terra konvertiert man eine Liste von Rastern zuerst in eine SpatRasterCollection (sprc)
tile_collection <- sprc(combined_tiles)
current_density <- mosaic(tile_collection, fun = "mean")

# Reclassify into deciles (1 = low probability, 10 = pinch-point/bottleneck)
vals <- values(current_density, na.rm = TRUE)
decile_breaks <- quantile(vals, probs = seq(0, 1, 0.1), na.rm = TRUE)
current_deciles <- classify(current_density, 
                            rcl = cbind(decile_breaks[-length(decile_breaks)], 
                                        decile_breaks[-1], 
                                        1:10),
                            include.lowest = TRUE) # Wichtig, damit der tiefste Wert nicht NA wird

writeRaster(current_density, file.path(OUT_DIR, "pinchpoint_current_density.tif"), overwrite = TRUE)
writeRaster(current_deciles, file.path(OUT_DIR, "pinchpoint_deciles.tif"), overwrite = TRUE)
cat("  Pinch-point maps saved.\n")


── 3. Circuitscape pinch-point analysis (tile approach) ──
[ Info: 2026-04-14 17:29:20 : Precision used: double
[ Info: 2026-04-14 17:29:20 : Reading maps
[ Info: 2026-04-14 17:29:22 : Resistance/Conductance map has 130484 nodes
[ Info: 2026-04-14 17:29:24 : Total number of pair solves = 1
[ Info: 2026-04-14 17:29:24 : Solving pair 1 of 1
[ Info: 2026-04-14 17:29:26 : Solver used: AMG accelerated by CG
[ Info: 2026-04-14 17:29:28 : Graph has 130275 nodes, 2 focal points and 1 connected components
[ Info: 2026-04-14 17:29:52 : Precision used: double
[ Info: 2026-04-14 17:29:52 : Reading maps
[ Info: 2026-04-14 17:29:53 : Resistance/Conductance map has 130484 nodes
[ Info: 2026-04-14 17:29:56 : Total number of pair solves = 1
[ Info: 2026-04-14 17:29:56 : Solving pair 1 of 1
[ Info: 2026-04-14 17:29:58 : Solver used: AMG accelerated by CG
[ Info: 2026-04-14 17:29:59 : Graph has 130284 nodes, 2 focal points and 1 connected components
  Tile 1/36 processed
[ Info: 2026-04-14 17:30:19 : Pr

## Least Cost Path

In [6]:
# --------------------------------------------- 4. CUMULATIVE COST CORRIDORS -
# Least-cost corridors for each graph edge pair → aggregate by cell minimum

cat("\n── 4. Cumulative cost corridors (least-cost paths) ──\n")

# Convert resistance to gdistance TransitionLayer
# Use 8-direction (queen's case) movement
library(gdistance)

# terra → RasterLayer for gdistance compatibility
resistance_rl <- raster::raster(resistance)

tr <- transition(resistance_rl, transitionFunction = function(x) 1 / mean(x),
                 directions = 8)
tr <- geoCorrection(tr, type = "c")   # correct for map distortion

# For each graph edge: compute cost distance from patch i, then corridor to j
corridor_stack <- list()

for (e in seq_len(nrow(edges))) {
  i <- edges$from[e]
  j <- edges$to[e]

  # Patch centroids as SpatialPoints
  pt_i <- sp::SpatialPoints(coords[i, , drop = FALSE],
                             proj4string = sp::CRS(st_crs(cores_sf)$proj4string))
  pt_j <- sp::SpatialPoints(coords[j, , drop = FALSE],
                             proj4string = sp::CRS(st_crs(cores_sf)$proj4string))

  # Accumulated cost surfaces from each patch centroid
  acc_i <- accCost(tr, pt_i)
  acc_j <- accCost(tr, pt_j)

  # Corridor = sum of both cost surfaces (standard least-cost corridor)
  corridor_ij <- raster::overlay(acc_i, acc_j, fun = function(a, b) a + b)

  corridor_stack[[e]] <- corridor_ij

  if (e %% 10 == 0 || e == nrow(edges))
    cat(sprintf("  Edge %d/%d done\n", e, nrow(edges)))
}

# Aggregate: cell-wise MINIMUM across all corridors (most favourable path)
cat("  Aggregating corridors (cell-wise minimum)...\n")
cum_cost_min <- Reduce(function(a, b) raster::overlay(a, b,
                        fun = function(x, y) pmin(x, y, na.rm = TRUE)),
                       corridor_stack)

cum_cost_terra <- rast(cum_cost_min)

# Reclassify into deciles (1 = preferred corridor, 10 = barrier)
vals_cc <- values(cum_cost_terra, na.rm = TRUE)
breaks_cc <- quantile(vals_cc, probs = seq(0, 1, 0.1), na.rm = TRUE)
cum_cost_deciles <- classify(cum_cost_terra,
                             rcl = cbind(breaks_cc[-length(breaks_cc)],
                                         breaks_cc[-1], 1:10))

writeRaster(cum_cost_terra,   file.path(OUT_DIR, "cumcost_raw.tif"),   overwrite = TRUE)
writeRaster(cum_cost_deciles, file.path(OUT_DIR, "cumcost_deciles.tif"), overwrite = TRUE)
cat("  Cumulative cost maps saved.\n")





── 4. Cumulative cost corridors (least-cost paths) ──
  Edge 10/572 done
  Edge 20/572 done
  Edge 30/572 done
  Edge 40/572 done
  Edge 50/572 done
  Edge 60/572 done
  Edge 70/572 done
  Edge 80/572 done
  Edge 90/572 done
  Edge 100/572 done
  Edge 110/572 done
  Edge 120/572 done
  Edge 130/572 done
  Edge 140/572 done
  Edge 150/572 done
  Edge 160/572 done
  Edge 170/572 done
  Edge 180/572 done
  Edge 190/572 done
  Edge 200/572 done
  Edge 210/572 done
  Edge 220/572 done
  Edge 230/572 done
  Edge 240/572 done
  Edge 250/572 done
  Edge 260/572 done
  Edge 270/572 done
  Edge 280/572 done
  Edge 290/572 done
  Edge 300/572 done
  Edge 310/572 done
  Edge 320/572 done
  Edge 330/572 done
  Edge 340/572 done
  Edge 350/572 done
  Edge 360/572 done
  Edge 370/572 done
  Edge 380/572 done
  Edge 390/572 done
  Edge 400/572 done
  Edge 410/572 done
  Edge 420/572 done
  Edge 430/572 done
  Edge 440/572 done
  Edge 450/572 done
  Edge 460/572 done
  Edge 470/572 done
  Edge 480/572

## Priority Map

In [7]:
# --------------------------------------------- 5. COMBINED PRIORITY MAP -----
# Unify pinch-point deciles + cumulative cost deciles into single priority map
# High priority (decile 10) = high current density AND low cumulative cost

cat("\n── 5. Combined priority map ──\n")

# Align grids
pp_aligned <- resample(current_deciles, cum_cost_deciles, method = "near")

# Invert cumulative cost deciles so high = preferred (low cost → high priority)
cc_inv <- 11 - cum_cost_deciles

# Equal-weight combination (mean), then re-decile
priority_raw <- (pp_aligned + cc_inv) / 2

vals_pr <- values(priority_raw, na.rm = TRUE)
breaks_pr <- quantile(vals_pr, probs = seq(0, 1, 0.1), na.rm = TRUE)
priority_deciles <- classify(priority_raw,
                             rcl = cbind(breaks_pr[-length(breaks_pr)],
                                         breaks_pr[-1], 1:10))

# Set impassable areas (max resistance) to priority 0
max_res <- global(resistance, "max", na.rm = TRUE)[[1]]
priority_deciles[resistance >= max_res] <- 0

writeRaster(priority_deciles, file.path(OUT_DIR, "priority_map_deciles.tif"),
            overwrite = TRUE)
cat("  Priority map saved.\n")





── 5. Combined priority map ──


Warning message:
[+] CRS do not match 
Warning message:
[mask] CRS do not match 


  Priority map saved.


In [8]:
# --------------------------------------------- 6. SUMMARY PLOT ---------------

cat("\n── 6. Generating summary figure ──\n")

rasters_to_plot <- list(
  "Pinch-Point Density (Deciles)"  = current_deciles,
  "Cumulative Cost (Deciles)"      = cum_cost_deciles,
  "Combined Priority"              = priority_deciles
)

png(file.path(OUT_DIR, "connectivity_summary.png"),
    width = 3600, height = 1400, res = 220)
par(mfrow = c(1, 3), mar = c(2, 2, 3, 4))
for (nm in names(rasters_to_plot)) {
  plot(rasters_to_plot[[nm]], main = nm,
       col = hcl.colors(10, "Zissou 1"),
       legend = TRUE, axes = FALSE)
  plot(st_geometry(cores_sf), add = TRUE,
       col = NA, border = "black", lwd = 0.8)
}
dev.off()

cat("\n✔ Pipeline complete. All outputs in:", OUT_DIR, "\n")
cat("
Output files:
  landscape_graph.png            – visual of graph edges + nodes
  graph_edges.csv                – edge list (from/to patch IDs + distance)
  pinchpoint_current_density.tif – raw Circuitscape current density mosaic
  pinchpoint_deciles.tif         – decile-classified pinch-point map
  cumcost_raw.tif                – raw minimum cumulative cost surface
  cumcost_deciles.tif            – decile-classified cumulative cost map
  priority_map_deciles.tif       – combined priority map (0 = impassable, 10 = highest)
  connectivity_summary.png       – 3-panel summary figure\n")


── 6. Generating summary figure ──

✔ Pipeline complete. All outputs in: ../data/processed/connectivity/ 

Output files:
  landscape_graph.png            – visual of graph edges + nodes
  graph_edges.csv                – edge list (from/to patch IDs + distance)
  pinchpoint_current_density.tif – raw Circuitscape current density mosaic
  pinchpoint_deciles.tif         – decile-classified pinch-point map
  cumcost_raw.tif                – raw minimum cumulative cost surface
  cumcost_deciles.tif            – decile-classified cumulative cost map
  priority_map_deciles.tif       – combined priority map (0 = impassable, 10 = highest)
  connectivity_summary.png       – 3-panel summary figure
